# mcp

> MCP server for fossick — search, fetch, read, hidden-API discovery, and browser automation as MCP tools, so Claude, Codex, and any other MCP client can drive the full toolkit.

In [ ]:
#| default_exp mcp

In [ ]:
#| hide
from nbdev.showdoc import *

The server wraps the same functions the Python API and CLI expose, over stdio. `mcp` ships as a dependency, so a plain install is all you need:

```sh
uv add fossick        # or: pip install fossick
fossick-mcp           # stdio server; --http for Streamable HTTP
```

In [ ]:
#| export
"""MCP server for fossick — search, fetch, read, hidden-API, and browser tools over stdio.
Run with `fossick-mcp` (the `mcp` package ships with fossick).

Docs: https://vedicreader.github.io/fossick/mcp.html.md"""

import json, sys
from fossick.core import (fetch as _fetch, to_md, crawl as _crawl, fetch_all as _fetch_all,
                          read_arxiv as _read_arxiv, lookup_doi as _lookup_doi,
                          read_yt as _read_yt, search_yt as _search_yt, download_yt as _download_yt,
                          read_gh_repo as _read_gh_repo, read_gh_file as _read_gh_file,
                          find_xhr as _find_xhr, replay_xhr as _replay_xhr,
                          paginate_api as _paginate_api, url2nb as _url2nb, pdf2nb as _pdf2nb)
from fossick.search import search as _search, google as _google, research as _research

try: from mcp.server.fastmcp import FastMCP
except ImportError as e:
    raise ImportError("fossick-mcp needs the `mcp` package — reinstall fossick "
                      "with `uv add fossick` or `pip install fossick`") from e

In [ ]:
#| export
mcp = FastMCP('fossick', instructions=(
    'Web toolkit. web_search/research for queries; fetch_page for URLs (heavy=JS, stealthy=anti-bot, '
    'session=logged-in Chrome, auto=escalate until not bot-blocked); read_arxiv/read_youtube/read_github_* '
    'for those sites; find_hidden_apis + replay_capture/paginate_api to turn a page into a JSON data feed; '
    'browse + page_* tools drive a persistent logged-in Chrome for interactive/authenticated flows.'))

def _trunc(s, n):
    'Clip `s` to `n` chars with an explicit truncation marker.'
    s = s or ''
    return s if not n or len(s) <= n else s[:n] + f'\n…[truncated {len(s)-n} of {len(s)} chars]'

def _jsonable(o):
    'Recursively convert result objects (L, Response, Path, ...) to JSON-serializable types.'
    if isinstance(o, dict): return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (str, int, float, bool)) or o is None: return o
    if hasattr(o, '__iter__'): return [_jsonable(x) for x in o]
    return str(o)

## Search and research

In [ ]:
#| export
@mcp.tool()
def web_search(query:str, n:int=10, category:str='text', region:str='us-en', google:bool=False) -> list:
    "Search the web via ddgs metasearch (no API key). category: text|images|news|videos|books. google=True gives real Google ranking via a stealth browser (slow — only when you need Google)."
    res = _google(query, n=n) if google else _search(query, category=category, max_results=n, region=region)
    return _jsonable(res)

@mcp.tool()
def research(query:str, n:int=5, google:bool=False, sel:str|None=None, chars:int=4000) -> dict:
    "Question -> cited answer: search, read the top n results in parallel (auto-escalating past bot walls), return {query, sources, digest} where digest is cited markdown, one ## section per source."
    return _jsonable(_research(query, n=n, engine='google' if google else 'search', sel=sel, chars=chars))

@mcp.tool()
def lookup_doi(title:str) -> str:
    "Return the doi.org URL for the first Crossref match on a paper title."
    return _lookup_doi(title) or "not found"

## Fetch and crawl

In [ ]:
#| export
@mcp.tool()
def fetch_page(url:str, sel:str|None=None, heavy:bool=False, stealthy:bool=False, session:bool=False,
               auto:bool=False, max_chars:int=8000) -> dict:
    "Fetch a URL as markdown. Always pass sel (CSS selector) to skip nav/ads. heavy=JS rendering; stealthy=anti-bot fetcher; session=reuse the logged-in debug Chrome's cookies; auto=escalate plain->heavy->stealthy->session (winning tier in result)."
    p = _fetch(url, sel=sel, heavy=heavy, stealthy=stealthy, session=session, auto=auto)
    return {'url': p['url'], 'status': p['status'], 'tier': getattr(p, 'tier', None),
            'markdown': _trunc(to_md(p), max_chars)}

@mcp.tool()
def fetch_pages(urls:list, sel:str|None=None, max_chars:int=4000) -> list:
    "Fetch many known URLs in parallel; returns [{url, status, markdown}] in the same order as urls."
    return [{'url': p['url'], 'status': p['status'], 'markdown': _trunc(to_md(p, sel=sel), max_chars)}
            for p in _fetch_all(urls, sel=sel)]

@mcp.tool()
def crawl_site(url:str, sel:str|None=None, follow_sel:str='a[href]', max_pages:int=10,
               same_domain:bool=True, heavy:bool=False, stealthy:bool=False, max_chars:int=4000) -> list:
    "Crawl from a start URL following follow_sel links (docs sites, blogs, multi-page content); returns [{url, status, markdown}] per page."
    pages = _crawl(url, sel=sel, follow_sel=follow_sel, max_pages=max_pages,
                   same_domain=same_domain, heavy=heavy, stealthy=stealthy)
    return [{'url': p['url'], 'status': p['status'], 'markdown': _trunc(to_md(p, sel=sel), max_chars)}
            for p in pages]

## Readers — arXiv, YouTube, GitHub, notebooks

In [ ]:
#| export
@mcp.tool()
def read_arxiv(url:str, include_source:bool=False, chars:int=8000, save_dir:str='.') -> dict:
    "arXiv paper (ID or any arXiv URL) -> {title, authors, published, summary, pdf_path}. include_source adds the full text (30-100k chars total — raise chars only when needed)."
    p = _read_arxiv(url, save_dir=save_dir)
    out = {k: _jsonable(v) for k, v in p.items() if k != 'source'}
    if include_source: out['source'] = _trunc(p.get('source'), chars)
    return out

@mcp.tool()
def read_youtube(url:str, chars:int=20000) -> dict:
    "YouTube video (URL or ID) -> metadata + full English transcript (in 'source')."
    v = {k: _jsonable(val) for k, val in dict(_read_yt(url)).items()}
    v['source'] = _trunc(v.get('source'), chars)
    return v

@mcp.tool()
def search_youtube(query:str, n:int=10) -> list:
    "Search YouTube; returns [{title, url, channel, duration, description}]."
    return _jsonable(_search_yt(query, n=n))

@mcp.tool()
def download_youtube(url:str, format:str='audio', save_dir:str='.') -> str:
    "Download YouTube audio or video. format: audio|video|any yt-dlp format string. Returns the saved file path."
    return str(_download_yt(url, format=format, save_dir=save_dir))

@mcp.tool()
def read_github_file(url:str, max_chars:int=20000) -> str:
    "Read a single file from a GitHub blob URL."
    return _trunc(_read_gh_file(url), max_chars)

@mcp.tool()
def read_github_repo(url:str, globs:list|None=None, limit:int|None=None, max_chars_per_file:int=8000) -> dict:
    "Read files from a GitHub repo (URL, SSH address, or local path) filtered by glob patterns (default: README*, pyproject.toml, *.py). Returns {path: content}."
    files = _read_gh_repo(url, globs=tuple(globs) if globs else None, limit=limit)
    return {str(k): _trunc(v, max_chars_per_file) for k, v in files.items()}

@mcp.tool()
def url_to_notebook(url:str, path:str|None=None) -> str:
    "Convert a URL (HTML page, PDF, or arXiv paper) to a Jupyter notebook; returns the notebook path."
    return str(_url2nb(url, nb_path=path))

@mcp.tool()
def pdf_to_notebook(src:str, path:str|None=None, ocr:str='auto') -> str:
    "Convert a PDF (URL or local path) to a notebook — one markdown cell per page. ocr: auto|on|off."
    return str(_pdf2nb(src, nb_path=path, ocr_selection=ocr))

## Hidden APIs

`find_hidden_apis` captures the JSON/XHR calls a page makes; each hit gets a `capture_id` that `replay_capture` re-issues as a fast plain-HTTP call (reusing the debug Chrome's cookies when the capture came from `session=True`). `paginate_api` replays a known endpoint across pages.

In [ ]:
#| export
_captures = []   # replayable request captures from the last find_hidden_apis/capture_network call

def _stash(caps):
    'Replace the stored captures with `caps`; their indices are the capture_ids.'
    _captures.clear(); _captures.extend(caps)

@mcp.tool()
def find_hidden_apis(url:str, pattern:str='*', session:bool=False, preview_chars:int=500) -> list:
    "Visit a page with a browser and capture the JSON/XHR API calls it makes (glob/regex pattern filters URLs). session=True captures through the logged-in debug Chrome. Each hit has a capture_id for replay_capture."
    hits = _find_xhr(url, pattern=pattern, session=session)
    _stash([h.get('capture') for h in hits])
    return [{'capture_id': i, 'url': h['url'], 'content_type': h.get('content_type'),
             'data_preview': _trunc(json.dumps(_jsonable(h.get('data')), default=str), preview_chars)}
            for i, h in enumerate(hits)]

@mcp.tool()
def replay_capture(capture_id:int, data:str|None=None, max_chars:int=8000) -> dict:
    "Re-issue a captured request (by capture_id from find_hidden_apis/capture_network) as a fast plain-HTTP call, reusing the browser's cookies. data overrides the request body."
    if not (0 <= capture_id < len(_captures)) or _captures[capture_id] is None:
        raise ValueError(f'no capture #{capture_id} — run find_hidden_apis or capture_network first')
    r = _replay_xhr(_captures[capture_id], data=data)
    try: body = r.json()
    except Exception: body = getattr(r, 'text', None) or getattr(r, 'body', None) or str(r)
    if not isinstance(body, str): body = json.dumps(_jsonable(body), default=str)
    return {'status': getattr(r, 'status', None), 'body': _trunc(body, max_chars)}

@mcp.tool()
def paginate_api(url:str, payload:dict|None=None, page_field:str='pageNumber', size_field:str='pageSize',
                 results_field:str|None=None, method:str='POST', max_pages:int=10, page_start:int=1,
                 max_items:int=500) -> list:
    "Paginate a JSON API, collecting all items across pages. payload is the base body (POST) or params (GET); page_field is the key incremented per page; results_field is auto-detected if None."
    items = _paginate_api(url, payload=payload, page_field=page_field, size_field=size_field,
                          results_field=results_field, method=method, max_pages=max_pages, page_start=page_start)
    return _jsonable(items[:max_items])

## Browser automation

These tools drive the persistent debug Chrome (launched headless on first use; cookies persist across runs, so log in once by hand and every later session stays authenticated). `browse` opens a page and returns a compact accessibility snapshot; the other `page_*` tools act on that page.

In [ ]:
#| export
_browser = {'page': None}   # current debug-Chrome page driven by browse/page_* tools

def _page():
    'The page opened by the last `browse` call.'
    if _browser['page'] is None: raise ValueError('no open page — call browse(url) first')
    return _browser['page']

@mcp.tool()
def browse(url:str, port:int=9223, full:bool=False) -> str:
    "Open a URL in the persistent debug Chrome and return a compact accessibility snapshot ('[#id] role \"name\"' per element). Later page_* tools act on this page. full=True includes non-interactive elements."
    from fossick.cdp import cdp_connect, syncy
    async def _run():
        cdp = await cdp_connect(port=port)
        return await cdp.open_page(url)
    _browser['page'] = pg = syncy(_run())
    return syncy(pg.snapshot(interactive=not full))

@mcp.tool()
def page_snapshot(full:bool=False) -> str:
    "Re-read the current page's accessibility snapshot (after JS/DOM changes)."
    from fossick.cdp import syncy
    return syncy(_page().snapshot(interactive=not full))

@mcp.tool()
def page_fill_form(fields:dict, submit:str|None=None) -> str:
    "Fill form fields on the current page by visible label ({label: value}; handles <select>), optionally click a submit button by label. Returns the post-action snapshot."
    from fossick.cdp import syncy
    return syncy(_page().fill_form(fields, submit=submit))

@mcp.tool()
def page_act(steps:list) -> dict:
    "Run a declarative flow on the current page. Steps (JSON arrays): ['goto',url] ['fill',label,value] ['click',label] ['select',label,option] ['wait',text] ['wait_sel',css] ['read',css] or ['read',css,label]. Returns {label: markdown} for every read step."
    from fossick.cdp import syncy
    return _jsonable(syncy(_page().act([tuple(s) for s in steps])))

@mcp.tool()
def page_markdown(sel:str|None=None, max_chars:int=8000) -> str:
    "Read the current page's live post-JS DOM as markdown, optionally narrowed by a CSS selector."
    from fossick.cdp import syncy
    return _trunc(syncy(_page().md(sel=sel)), max_chars)

@mcp.tool()
def capture_network(url:str, pattern:str='.*', tail:int=3, port:int=9223, preview_chars:int=300) -> list:
    "Navigate the debug Chrome to a URL and capture outgoing network requests matching pattern (listens tail seconds after load). Each request gets a capture_id for replay_capture."
    from fossick.cdp import cdp_connect, syncy
    async def _run():
        cdp = await cdp_connect(port=port)
        return await cdp.calls(url=url, pattern=pattern, tail=tail)
    vals = list(syncy(_run()).values())
    _stash(vals)
    return [{'capture_id': i, 'url': r['url'], 'method': r.get('method'),
             'response_preview': _trunc(str(r.get('response_body') or ''), preview_chars)}
            for i, r in enumerate(vals)]

In [ ]:
#| export
def main():
    "Entry point for the `fossick-mcp` console script. stdio by default; pass --http for Streamable HTTP."
    mcp.run(transport='streamable-http' if '--http' in sys.argv[1:] else 'stdio')

## Connect a client

**Claude Code**

```sh
claude mcp add fossick -- uvx --from fossick fossick-mcp
```

**Codex** (`~/.codex/config.toml`)

```toml
[mcp_servers.fossick]
command = "uvx"
args = ["--from", "fossick", "fossick-mcp"]
```

**Claude Desktop** (`claude_desktop_config.json`)

```json
{"mcpServers": {"fossick": {"command": "uvx", "args": ["--from", "fossick", "fossick-mcp"]}}}
```

If fossick is already a dependency of your project, `uv run fossick-mcp` reuses the project venv instead of a second install.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()